# Phase 2 — Step 1: Data Pipeline
**Dissertation: Construct Validity in Transformer-Based AES | Student ID: 5720570**

Builds the stratified train/val/test split and BERT tokenisation pipeline.
- 80 / 10 / 10 split stratified by prompt AND score
- Head+tail truncation for 512-token BERT limit (128 head + 382 tail + 2 special tokens)
- PyTorch Dataset and DataLoader ready for Phase 2 Step 2 (BERT fine-tuning)

In [10]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import BertTokenizerFast
from collections import Counter
import os, warnings
warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.10.0+cu128
GPU available: True
GPU: Tesla T4


In [11]:
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

In [12]:
# ── Device setup — referenced in all subsequent cells ─────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU name:   {torch.cuda.get_device_name(0)}')
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU name:   Tesla T4
GPU memory: 15.6 GB


## Step 1 — Load Dataset (Essay Level)
Collapse discourse-level rows to one row per essay using `essay_id`.

In [13]:
DATA_PATH = '/kaggle/input/datasets/shravaniuk/persuade-corpus-2-0-tarin-test-b'

df_raw = pd.read_csv(f'{DATA_PATH}/persuade_corpus_2.0_train.csv')

# Collapse to essay level — keep first occurrence per essay_id
KEEP_COLS = ['essay_id', 'full_text', 'holistic_essay_score',
             'prompt_name', 'grade_level', 'gender',
             'race_ethnicity', 'ell_status',
             'student_disability_status', 'economically_disadvantaged', 'task']

df = (df_raw[KEEP_COLS]
        .drop_duplicates(subset='essay_id')
        .reset_index(drop=True))

# Rename score column for brevity
df = df.rename(columns={'holistic_essay_score': 'score'})

print(f"Essays loaded:  {len(df):,}")
print(f"Score range:    {df['score'].min()} – {df['score'].max()}")
print(f"Prompts:        {df['prompt_name'].nunique()}")
print(f"Null full_text: {df['full_text'].isna().sum()}")

Essays loaded:  15,593
Score range:    1 – 6
Prompts:        15
Null full_text: 0


## Step 2 — Stratified Train / Validation / Test Split
Stratification key = `prompt_name + "_" + score` ensures every split
contains all 15 prompts × 6 score levels proportionally.
Split ratio: **80% train | 10% val | 10% test**

In [14]:
# Combined stratification label
df['strat_key'] = df['prompt_name'].str.replace(' ', '_') + '_' + df['score'].astype(str)

# Remove strat groups with < 2 samples (cannot split)
key_counts = df['strat_key'].value_counts()
valid_keys = key_counts[key_counts >= 2].index
df_valid = df[df['strat_key'].isin(valid_keys)].copy()
df_removed = df[~df['strat_key'].isin(valid_keys)]

print(f"Total essays:        {len(df):,}")
print(f"Usable for split:    {len(df_valid):,}")
print(f"Removed (rare keys): {len(df_removed):,}")
print(f"Unique strat keys:   {df_valid['strat_key'].nunique()}")

Total essays:        15,593
Usable for split:    15,591
Removed (rare keys): 2
Unique strat keys:   86


In [15]:
# Step 1: split off 20% for val+test
df_train, df_temp = train_test_split(
    df_valid,
    test_size=0.20,
    random_state=42,
    stratify=df_valid['strat_key']
)

# Step 2: split temp 50/50 → val and test
df_val, df_test = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=42,
    stratify= None
)

# Drop helper column
for split in [df_train, df_val, df_test]:
    split.drop(columns=['strat_key'], inplace=True)

print(f"Train: {len(df_train):,} essays ({len(df_train)/len(df_valid)*100:.1f}%)")
print(f"Val:   {len(df_val):,}  essays ({len(df_val)/len(df_valid)*100:.1f}%)")
print(f"Test:  {len(df_test):,}  essays ({len(df_test)/len(df_valid)*100:.1f}%)")

Train: 12,472 essays (80.0%)
Val:   1,559  essays (10.0%)
Test:  1,560  essays (10.0%)


In [16]:
print("Score distribution across splits:")
print(f"{'Score':<8} {'Train%':>8} {'Val%':>8} {'Test%':>8}")
print("-" * 36)
for score in sorted(df_train['score'].unique()):
    t = (df_train['score'] == score).mean() * 100
    v = (df_val['score']   == score).mean() * 100
    te= (df_test['score']  == score).mean() * 100
    print(f"{score:<8} {t:>7.1f}% {v:>7.1f}% {te:>7.1f}%")

print()
print("Prompt coverage (all splits should have 15):")
print(f"  Train: {df_train['prompt_name'].nunique()} prompts")
print(f"  Val:   {df_val['prompt_name'].nunique()} prompts")
print(f"  Test:  {df_test['prompt_name'].nunique()} prompts")

Score distribution across splits:
Score      Train%     Val%    Test%
------------------------------------
1            3.9%     3.8%     4.0%
2           20.6%    21.6%    19.7%
3           31.7%    31.0%    32.4%
4           26.9%    25.8%    28.0%
5           13.4%    13.9%    12.8%
6            3.5%     4.0%     2.9%

Prompt coverage (all splits should have 15):
  Train: 15 prompts
  Val:   15 prompts
  Test:  15 prompts


## Step 3 — BERT Tokenisation (Head + Tail Truncation)
**Strategy:** For essays exceeding 512 tokens, keep:
- First 128 tokens (preserves thesis / introduction)
- Last 383 tokens (preserves conclusion)
- + `[CLS]` and `[SEP]` = **512 total**

This preserves the most construct-relevant text at both ends of the essay.

In [17]:
TOKENIZER_NAME = 'bert-base-uncased'
MAX_LEN = 512
HEAD_LEN = 128
TAIL_LEN = MAX_LEN - HEAD_LEN   # 384

tokenizer = BertTokenizerFast.from_pretrained(TOKENIZER_NAME)
print(f"Tokeniser loaded: {TOKENIZER_NAME}")
print(f"Vocab size: {tokenizer.vocab_size:,}")
print(f"Head tokens: {HEAD_LEN} | Tail tokens: {TAIL_LEN} | Max: {MAX_LEN}")


def head_tail_tokenize(text, tokenizer, max_len=512, head=128, tail=384):
    """
    Tokenise with head+tail truncation.
    Returns input_ids, attention_mask as tensors of length max_len.
    """
    import transformers
    transformers.logging.set_verbosity_error()
    tokens = tokenizer.encode(
    str(text),
    add_special_tokens=False,
    truncation=False
)

    if len(tokens) <= (max_len - 2):
        # Fits within limit — standard tokenisation
        encoding = tokenizer(
            str(text),
            max_length=max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
    else:
        # Apply head + tail truncation
        # -2 accounts for [CLS] and [SEP]
        head_tokens = tokens[:head - 1]       # 127 head tokens
        tail_tokens = tokens[-(tail - 1):]    # 383 tail tokens
        combined = head_tokens + tail_tokens  # 510 tokens

        # Manually build input_ids with special tokens
        cls_id = tokenizer.cls_token_id
        sep_id = tokenizer.sep_token_id
        pad_id = tokenizer.pad_token_id

        input_ids = [cls_id] + combined + [sep_id]
        # Pad if needed
        padding = [pad_id] * (max_len - len(input_ids))
        input_ids = input_ids + padding
        attention_mask = [1] * (len(combined) + 2) + [0] * len(padding)

        encoding = {
            'input_ids':      torch.tensor([input_ids]),
            'attention_mask': torch.tensor([attention_mask])
        }

    return (
        encoding['input_ids'].squeeze(0),
        encoding['attention_mask'].squeeze(0)
    )

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokeniser loaded: bert-base-uncased
Vocab size: 30,522
Head tokens: 128 | Tail tokens: 384 | Max: 512


In [18]:
# Test on one short and one long essay
short_text = "This is a short essay."
long_text   = " ".join(["word"] * 800)  # Simulated long essay

ids_s, mask_s = head_tail_tokenize(short_text, tokenizer)
ids_l, mask_l = head_tail_tokenize(long_text,  tokenizer)

print(f"Short essay → input_ids shape: {ids_s.shape}, attention sum: {mask_s.sum().item()}")
print(f"Long essay  → input_ids shape: {ids_l.shape}, attention sum: {mask_l.sum().item()}")
assert ids_s.shape[0] == 512, "Shape mismatch on short essay"
assert ids_l.shape[0] == 512, "Shape mismatch on long essay"
print("Tokenisation check passed — all outputs are length 512")

Short essay → input_ids shape: torch.Size([512]), attention sum: 8
Long essay  → input_ids shape: torch.Size([512]), attention sum: 512
Tokenisation check passed — all outputs are length 512


## Step 4 — PyTorch Dataset and DataLoaders
Wraps each split into a `torch.utils.data.Dataset`.
Scores are normalised to [0, 1] for MSE regression training,
then converted back to 1–6 scale for QWK evaluation.

In [19]:
class EssayDataset(Dataset):
    """
    PyTorch Dataset for PERSUADE 2.0 essay scoring.
    Tokenises on the fly using head+tail strategy.
    Score is normalised: label = (score - 1) / 5  → [0.0, 1.0]
    """
    def __init__(self, dataframe, tokenizer, max_len=512):
        self.texts  = dataframe['full_text'].tolist()
        self.scores = dataframe['score'].tolist()
        self.ids    = dataframe['essay_id'].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        input_ids, attention_mask = head_tail_tokenize(
            self.texts[idx], self.tokenizer, self.max_len
        )
        # Normalise score to [0, 1] for regression
        label = torch.tensor((self.scores[idx] - 1) / 5.0, dtype=torch.float)

        return {
            'input_ids':      input_ids,
            'attention_mask': attention_mask,
            'label':          label,
            'score':          torch.tensor(self.scores[idx], dtype=torch.long),
            'essay_id':       self.ids[idx]
        }


# Instantiate datasets
train_dataset = EssayDataset(df_train, tokenizer)
val_dataset   = EssayDataset(df_val,   tokenizer)
test_dataset  = EssayDataset(df_test,  tokenizer)

print(f"Train dataset: {len(train_dataset):,} essays")
print(f"Val dataset:   {len(val_dataset):,}  essays")
print(f"Test dataset:  {len(test_dataset):,}  essays")

Train dataset: 12,472 essays
Val dataset:   1,559  essays
Test dataset:  1,560  essays


In [20]:
BATCH_SIZE = 16  # Safe for T4 GPU with BERT-base
NUM_WORKERS = 2

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# Verify one batch
batch = next(iter(train_loader))
print(f"Batch input_ids shape:      {batch['input_ids'].shape}")
print(f"Batch attention_mask shape: {batch['attention_mask'].shape}")
print(f"Batch labels (normalised):  {batch['label'][:4].tolist()}")
print(f"Batch scores (original):    {batch['score'][:4].tolist()}")
print(f"DataLoader verified — batch size {BATCH_SIZE}")

Batch input_ids shape:      torch.Size([16, 512])
Batch attention_mask shape: torch.Size([16, 512])
Batch labels (normalised):  [0.6000000238418579, 0.20000000298023224, 0.20000000298023224, 0.800000011920929]
Batch scores (original):    [4, 2, 2, 5]
DataLoader verified — batch size 16


## Step 5 — Save Splits for Phase 2 Step 2
Saves the three CSV splits so BERT training can reload them without repeating the split logic.

In [21]:
df_train.to_csv('persuade_train_split.csv', index=False)
df_val.to_csv(  'persuade_val_split.csv',   index=False)
df_test.to_csv( 'persuade_test_split.csv',  index=False)

print("Files saved:")
print(f"  persuade_train_split.csv — {len(df_train):,} rows")
print(f"  persuade_val_split.csv   — {len(df_val):,}  rows")
print(f"  persuade_test_split.csv  — {len(df_test):,}  rows")

Files saved:
  persuade_train_split.csv — 12,472 rows
  persuade_val_split.csv   — 1,559  rows
  persuade_test_split.csv  — 1,560  rows


In [22]:
print("=" * 55)
print("PHASE 2 STEP 1 — DATA PIPELINE COMPLETE")
print("=" * 55)
print(f"\n  Train essays:    {len(df_train):,}")
print(f"  Val essays:      {len(df_val):,}")
print(f"  Test essays:     {len(df_test):,}")
print(f"  Tokeniser:       {TOKENIZER_NAME}")
print(f"  Max length:      {MAX_LEN} tokens")
print(f"  Head / Tail:     {HEAD_LEN} / {TAIL_LEN} tokens")
print(f"  Batch size:      {BATCH_SIZE}")
print(f"  GPU ready:       {torch.cuda.is_available()}")
print(f"\n  Score normalisation: (score - 1) / 5 → [0.0, 1.0]")
print(f"  Loss function:       MSE (regression)")
print(f"  Evaluation metric:   QWK (target ≥ 0.75)")
print()
print("  NEXT → Phase 2 Step 2: BERT Fine-Tuning")
print("=" * 55)

PHASE 2 STEP 1 — DATA PIPELINE COMPLETE

  Train essays:    12,472
  Val essays:      1,559
  Test essays:     1,560
  Tokeniser:       bert-base-uncased
  Max length:      512 tokens
  Head / Tail:     128 / 384 tokens
  Batch size:      16
  GPU ready:       True

  Score normalisation: (score - 1) / 5 → [0.0, 1.0]
  Loss function:       MSE (regression)
  Evaluation metric:   QWK (target ≥ 0.75)

  NEXT → Phase 2 Step 2: BERT Fine-Tuning
